# Kubernetes Basics — Pods, Logs & Debugging

**[website version](https://training.nrp-nautilus.io/cms-hats/2_kubernetes_basics.html)** — run cells with **Shift+Enter**.

The rest of this training runs Kubernetes Jobs for you — you `kubectl apply` a manifest and read the logs. Before that, it helps to see the basic unit those Jobs are built from: a **pod**, and get comfortable with the handful of commands you'll use on every pod and Job for the rest of the day.

**In this exercise you will:**

1. Launch a single pod from a YAML manifest.
2. Read its logs.
3. Run a one-off command inside it, then open an interactive shell.
4. Practice the two commands you reach for the moment something looks wrong — `describe` and `get events`.
5. Clean up.

Everything here is deliberately small — one pod, no PVC, no image to build — so the mechanics stay visible. The jet-classifier exercise later in this training reuses every command you learn here, just aimed at a Job instead of a bare pod.

## One-time hub setup

If you already did this earlier in the session, skip ahead. Otherwise, do it now — the rest of the training assumes it's already done, including the grid certificate needed by the last lesson.

**If you're on the Analysis Hub:**

**🖥️ Terminal step** — open a terminal in JupyterLab (**File → New → Terminal**). `grid-kube-setup` and the `kubectl` login may already be done if you followed the setup page; running them again is harmless:

```bash
grid-kube-setup
kubectl get pods -n us-cms   # triggers a device-code login if needed
```

Then the grid certificate, needed later in [CMS Data Access on NRP](5_cms_data.ipynb) — both prompt for a password/pass phrase, so they need a real terminal too:

```bash
grid-cert-import
grid-proxy-init
```

Also set an environment variable in the terminal
```bash
export USER=<changeme> 
```
This will help when running some interactive commands later. Keep this terminal open or you will have to do this step again.

Full walkthroughs with expected output: [Get kubectl working in the hub terminal](../../lessons/0_setup.md) on the setup page, and [Setting up your grid certificate](5_cms_data.ipynb) in CMS Data Access.

**If you're on your own machine:** skip the block above — `grid-kube-setup`, `grid-cert-import`, and `grid-proxy-init` are Analysis Hub tools with no local install. You'll pick this up when you switch to the hub for CMS Data Access.

**Everyone**, set a short username once and reuse it for every command in this training — using the cell below.

## ⚙️ Set your username

Edit `USER` below, then run the cell.

In [ ]:
export USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/cms-hats/workspace
if [ "$USER" = changeme ]; then echo "⚠️  Edit USER above first, then re-run"; else
  cp yamls/pod-basics.yaml /tmp/pod-basics-${USER}.yaml
  perl -pi -e 's/<username>/$ENV{USER}/g' /tmp/pod-basics-${USER}.yaml
  echo "✅ /tmp/pod-basics-${USER}.yaml ready"
fi


## Launch a pod

`yamls/pod-basics.yaml` is deliberately minimal:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: pod-basics-<username>
  namespace: us-cms
spec:
  containers:
  - name: mypod
    image: ubuntu:22.04
    command: ["sh", "-c", "echo 'Hello from NRP!' && sleep 3600"]
    resources:
      limits:   { memory: 100Mi, cpu: 100m }
      requests: { memory: 100Mi, cpu: 100m }
```

In [ ]:
kubectl apply -n us-cms -f /tmp/pod-basics-${USER}.yaml


In [ ]:
kubectl wait --for=condition=Ready pod/pod-basics-${USER} -n us-cms --timeout=60s


In [ ]:
kubectl get pods -n us-cms


## Read the logs

We sleep briefly first — `Ready` means the container is running, not that it has necessarily flushed its first line of output yet.

In [ ]:
sleep 5
kubectl logs pod-basics-${USER} -n us-cms


<details>
<summary>Expected output</summary>

```text
pod/pod-basics-<username> created

NAME                        READY   STATUS    RESTARTS   AGE
pod-basics-<username>       1/1     Running   0          8s

Hello from NRP!
```
</details>

## Run commands inside the pod

In [ ]:
kubectl exec pod-basics-${USER} -n us-cms -- echo 'Command executed successfully'


**🖥️ Terminal step** — interactive: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-D to exit.

```bash
kubectl exec -it pod-basics-${USER} -n us-cms -- /bin/bash
```

## The debugging trio: describe, events, previous logs

When something doesn't behave the way you expect:

In [ ]:
kubectl describe pod pod-basics-${USER} -n us-cms


In [ ]:
kubectl get events -n us-cms --field-selector involvedObject.name=pod-basics-${USER} --sort-by=.metadata.creationTimestamp


`describe` shows scheduling decisions and container state; `get events` (filtered to just this pod with `--field-selector`) shows its scheduling timeline — pulling the image, mounting volumes, starting the container. Drop the `--field-selector` to see every event in the namespace instead. For a **crashlooping** pod, add `--previous` to read the dead container's logs — `kubectl logs <pod> -n us-cms --previous`. On a healthy pod it just says *"previous terminated container not found"* — that's expected, not an error.

## Clean up

In [ ]:
kubectl delete pod pod-basics-${USER} -n us-cms


## Taints, tolerations, and node affinity

NRP is a heterogeneous shared cluster — pools of nodes get reserved for specific projects or trainings, and are marked off with a **taint** so nobody else's workload accidentally lands there.

| Primitive | Lives on | Asks the question |
| --- | --- | --- |
| Node label | Node | "What is this node?" |
| `nodeSelector` / `nodeAffinity` | Pod | "Which nodes am I willing to land on?" |
| Taint | Node | "Who is allowed to land here?" |
| Toleration | Pod | "I have permission to land on those tainted nodes." |

Labels + affinity are an **attraction**; taints + tolerations are a **repulsion**. Landing on a reserved pool on purpose usually needs both: a toleration for permission, plus an affinity rule so the scheduler actually picks one of those nodes instead of just tolerating them in passing.

**⚠️ Only add a toleration when you're told to**

A toleration doesn't grant extra cluster-wide permissions, but it does let your pod schedule onto nodes someone deliberately set aside — usually for a specific project's reservation or limited/contended hardware. Adding a toleration to a pod spec just because it won't schedule, without knowing *why* the node is tainted, is how you end up occupying capacity that isn't meant for you.

Only add a toleration when a lesson or your namespace admin explicitly gives you the block to use — like the jet classifier YAMLs later in this training already do. If `kubectl describe pod` shows something like `0/N nodes are available: N node(s) had untolerated taint` and you *don't* have a specific toleration you were told to use, that's the scheduler correctly telling you those nodes aren't for you — the fix is to find capacity elsewhere, not to copy a toleration you found by searching around.

The jet classifier training, analysis, and sweep Jobs later in this training pull a custom image and need GPU-backed nodes reserved for this training, so their manifests include both:

```yaml
spec:
  tolerations:
  - key: nautilus.io/reservation
    operator: Equal
    value: nrp
    effect: NoSchedule
  affinity:
    nodeAffinity:
      preferredDuringSchedulingIgnoredDuringExecution:
      - weight: 100
        preference:
          matchExpressions:
          - key: nrp-training
            operator: In
            values: ["true"]
```

`preferredDuringSchedulingIgnoredDuringExecution` is a **soft** hint — the scheduler picks a reserved node if one's free but won't strand your pod if all of them are busy. `pod-basics.yaml` (this lesson) and the PVC-browser pod used later to copy results don't have this block at all — they don't pull the custom training image and run fine on any regular node, so there's nothing to tolerate.

---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash ~/cms-hats/workspace/check.sh 2
